In [38]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [30]:
import json
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

def search(query):
        boost_dict = {"question": 3.0, "section": 0.5}
        filter_dict = {"course": 'llm-zoomcamp'}

        return sqlite_index.search(
            query,
            num_results=5,
            boost_dict=boost_dict,
            filter_dict=filter_dict
        )

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [31]:

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)



question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="openai/gpt-oss-120b",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"Can I join the course enrollment eligibility"}


In [32]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.First perform search and then perform same search again



At the end, ask if there are other areas that the user wants to explore.
at the end say amigos
""".strip()

In [33]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]
    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model="openai/gpt-oss-120b",
            input=messages,
            tools=[search_tool],
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer=item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    return last_answer

In [34]:
agent_loop(instructions, "How do I run Olama locally?")


iteration #1...
function_call: search {"query":"Olama run locally"}
iteration #2...
function_call: search {"query":"Olama"}
iteration #3...
function_call: search {"query":"Ollama run locally"}
iteration #4...
ASSISTANT:
**Running Ollama locally**

Below is a quick‑start guide that walks you through installing Ollama on your machine, starting the local server, pulling a model, and using it from the command line or Python.

---

### 1️⃣ Install Ollama

| OS | Installation command / steps |
|----|------------------------------|
| **macOS** | Download the `.pkg` from <https://ollama.com/download> and run it. |
| **Windows** | Download the `.msi` from the same page and run the installer. |
| **Linux** | Run the one‑liner in a terminal: <br>`curl -fsSL https://ollama.com/install.sh \| sh` |

> After the installer finishes, you should be able to type `ollama` in a terminal and see the help output.

---

### 2️⃣ Start a model (the “run” command)

The simplest way to launch a model locally is:


'**Running\u202fOllama\u202flocally**\n\nBelow is a quick‑start guide that walks you through installing Ollama on your machine, starting the local server, pulling a model, and using it from the command line or Python.\n\n---\n\n### 1️⃣ Install Ollama\n\n| OS | Installation command / steps |\n|----|------------------------------|\n| **macOS** | Download the `.pkg` from <https://ollama.com/download> and run it. |\n| **Windows** | Download the `.msi` from the same page and run the installer. |\n| **Linux** | Run the one‑liner in a terminal: <br>`curl -fsSL https://ollama.com/install.sh \\| sh` |\n\n> After the installer finishes, you should be able to type `ollama` in a terminal and see the help output.\n\n---\n\n### 2️⃣ Start a model (the “run” command)\n\nThe simplest way to launch a model locally is:\n\n```bash\nollama run llama3\n```\n\nWhat happens:\n\n1. **Download** – Ollama pulls the requested model (e.g., `llama3`) from the Ollama registry (≈\u202f4\u202fGB for LLaMA\u202f3).  \n

In [35]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")


iteration #1...
function_call: search {"query":"Can I still join the course after it started?"}
iteration #2...
ASSISTANT:
Yes—you can still join the course even if you discovered it after it has started. The only catch is that, to earn a certificate, you’ll need to submit your capstone project while the submission window is still open (so that you can also complete the required peer‑reviews).

If you have any other questions about enrollment, the curriculum, project deadlines, or anything else, just let me know!  

¡Amigos!


'Yes—you can still join the course even if you discovered it after it has started. The only catch is that, to earn a certificate, you’ll need to submit your capstone project while the submission window is still open (so that you can also complete the required peer‑reviews).\n\nIf you have any other questions about enrollment, the curriculum, project deadlines, or anything else, just let me know!  \n\n¡Amigos!'

In [39]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"Queen's Gambit definition"}
iteration #2...
function_call: search {"query":"Queen's Gambit chess opening definition"}
iteration #3...
ASSISTANT:
**The Queen’s Gambit** is one of the oldest and most famous openings in chess.  

---

## 1. The basic move order  

| Move | White | Black |
|------|-------|-------|
| 1    | **d4** | … |
| 2    | **c4** | **d5** |
| 3    | **cxd5** | **c6** (or **e6**, **Nf6**, …) |

* After 1 d4 d5 2 c4 White offers the pawn on c4.  
* By playing 2…d5, Black accepts the “gift” of a pawn when White captures on d5 (3 cxd5).  
* The opening is called a *gambit* because White willingly gives up a pawn (or at least a pawn‑center) in exchange for rapid piece development and long‑term pressure on the centre.

---

## 2. Why play it? (Strategic ideas)

| White’s aims | Black’s aims |
|--------------|--------------|
| • Obtain a strong **central pawn duo** on d4 – e4 (often after playing e3 and then e4). <br>• Open lin

'**The Queen’s\u202fGambit** is one of the oldest and most famous openings in chess.  \n\n---\n\n## 1. The basic move order  \n\n| Move | White | Black |\n|------|-------|-------|\n| 1    | **d4** | … |\n| 2    | **c4** | **d5** |\n| 3    | **cxd5** | **c6** (or **e6**, **Nf6**, …) |\n\n* After 1\u202fd4\u202fd5 2\u202fc4 White offers the pawn on c4.  \n* By playing 2…d5, Black accepts the “gift” of a pawn when White captures on d5 (3\u202fcxd5).  \n* The opening is called a *gambit* because White willingly gives up a pawn (or at least a pawn‑center) in exchange for rapid piece development and long‑term pressure on the centre.\n\n---\n\n## 2. Why play it? (Strategic ideas)\n\n| White’s aims | Black’s aims |\n|--------------|--------------|\n| • Obtain a strong **central pawn duo** on d4\u202f–\u202fe4 (often after playing e3 and then e4). <br>• Open lines for the **Queenside bishop** (the “c‑bishop”) and the **queen**. <br>• Gain **piece activity** and a lead in development. | • Hold o

In [40]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
ASSISTANT:
I’m sorry, but I don’t have information on that topic in the course FAQ. Is there anything else related to the course that I can help you with?


'I’m sorry, but I don’t have information on that topic in the course FAQ. Is there anything else related to the course that I can help you with?'